# Install packages

In [156]:
!pip install -q -U transformers datasets peft accelerate scikit-learn sentencepiece
!pip install -U "torchao>=0.16.0"

# Imports and configuration

In [157]:
import math
import time
import random
import numpy as np

import torch
import torch.nn as nn
from torch.optim import Adam

from datasets import load_dataset
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    get_cosine_schedule_with_warmup,
)

from peft import LoraConfig, get_peft_model

In [158]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
# EXPERIMENT RESULTS

# all_results = {}

# Model

In [159]:
MODEL_NAME = "t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("Model loaded.")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Model loaded.


# Hyperparameters

In [160]:
REFERENCE_RANK = 8

MIN_RANK = 4
MAX_RANK = 32

N_PROBE_BATCHES = 64

PEAK_LR = 1e-4

BATCH_SIZE = 32

NUM_EPOCHS = 1

WARMUP_RATIO = 0.03

GAMMA = 5e-2

LORA_ALPHA = 32

MAX_LENGTH = 128

BETA1 = 0.9
BETA2 = 0.999
WEIGHT_DECAY = 0.0

B_LR_MULTIPLIER = 16.0

# Load MRPC

In [161]:
# DATASET CONFIGURATION

DATASET_NAME = "cola"

# Available:
#
# "mnli"
# "sst2"
# "cola"
# "qnli"
# "mrpc"
# "rte"

TASK_CONFIGS = {

    "mnli": {
        "dataset": "nyu-mll/glue",
        "subset": "mnli",
        "sentence1": "premise",
        "sentence2": "hypothesis",
        "label_names": {
            0: "entailment",
            1: "neutral",
            2: "contradiction",
        },
        "num_labels": 3,
        "metric": "accuracy",
    },

    "sst2": {
        "dataset": "nyu-mll/glue",
        "subset": "sst2",
        "sentence1": "sentence",
        "sentence2": None,
        "label_names": {
            0: "negative",
            1: "positive",
        },
        "num_labels": 2,
        "metric": "accuracy",
    },

    "cola": {
        "dataset": "nyu-mll/glue",
        "subset": "cola",
        "sentence1": "sentence",
        "sentence2": None,
        "label_names": {
            0: "unacceptable",
            1: "acceptable",
        },
        "num_labels": 2,
        "metric": "matthews",
    },

    "qnli": {
        "dataset": "nyu-mll/glue",
        "subset": "qnli",
        "sentence1": "question",
        "sentence2": "sentence",
        "label_names": {
            0: "entailment",
            1: "not_entailment",
        },
        "num_labels": 2,
        "metric": "accuracy",
    },

    "mrpc": {
        "dataset": "nyu-mll/glue",
        "subset": "mrpc",
        "sentence1": "sentence1",
        "sentence2": "sentence2",
        "label_names": {
            0: "not_equivalent",
            1: "equivalent",
        },
        "num_labels": 2,
        "metric": "accuracy",
    },

    "rte": {
        "dataset": "nyu-mll/glue",
        "subset": "rte",
        "sentence1": "sentence1",
        "sentence2": "sentence2",
        "label_names": {
            0: "not_entailment",
            1: "entailment",
        },
        "num_labels": 2,
        "metric": "accuracy",
    },
}


config = TASK_CONFIGS[DATASET_NAME]

print("Dataset:", DATASET_NAME)
print("Configuration:", config)

Dataset: cola
Configuration: {'dataset': 'nyu-mll/glue', 'subset': 'cola', 'sentence1': 'sentence', 'sentence2': None, 'label_names': {0: 'unacceptable', 1: 'acceptable'}, 'num_labels': 2, 'metric': 'matthews'}


In [162]:
# LOAD DATASET

raw_dataset = load_dataset(
    config["dataset"],
    config["subset"],
)

print(raw_dataset)

cola/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  251kB            

cola/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

cola/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 37.6kB            

cola/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

cola/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 37.7kB            

cola/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8551 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1043 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1063 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})


In [165]:
LABEL_NAMES = config["label_names"]

# Generic preprocessing

In [166]:
# GENERIC GLUE PREPROCESSING

LABEL_NAMES = config["label_names"]


def preprocess_glue(example):

    sentence1 = example[
        config["sentence1"]
    ]

    sentence2_key = config["sentence2"]

    if sentence2_key is not None:

        sentence2 = example[
            sentence2_key
        ]

        input_text = (
            f"{DATASET_NAME} "
            f"sentence1: {sentence1} "
            f"sentence2: {sentence2}"
        )

    else:

        input_text = (
            f"{DATASET_NAME} "
            f"sentence: {sentence1}"
        )

    target_text = LABEL_NAMES[
        example["label"]
    ]

    model_inputs = tokenizer(
        input_text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        target_text,
        max_length=8,
        truncation=True,
        padding="max_length",
    )

    labels["input_ids"] = [
        token
        if token != tokenizer.pad_token_id
        else -100
        for token in labels["input_ids"]
    ]

    model_inputs["labels"] = (
        labels["input_ids"]
    )

    return model_inputs

In [167]:
# TRAIN / VALIDATION SPLITS

tokenized_train = raw_dataset["train"].map(
    preprocess_glue,
    remove_columns=raw_dataset["train"].column_names,
)


if DATASET_NAME == "mnli":

    validation_split = "validation_matched"

else:

    validation_split = "validation"


tokenized_validation = raw_dataset[
    validation_split
].map(
    preprocess_glue,
    remove_columns=raw_dataset[
        validation_split
    ].column_names,
)

Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

# DataLoader

In [168]:
# DATALOADERS

def collate_fn(batch):

    return {
        "input_ids": torch.tensor(
            [x["input_ids"] for x in batch],
            dtype=torch.long,
        ),

        "attention_mask": torch.tensor(
            [x["attention_mask"] for x in batch],
            dtype=torch.long,
        ),

        "labels": torch.tensor(
            [x["labels"] for x in batch],
            dtype=torch.long,
        ),
    }


train_dataloader = DataLoader(
    tokenized_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

eval_dataloader = DataLoader(
    tokenized_validation,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

In [169]:
# GROUND-TRUTH LABELS

true_labels_text = [
    LABEL_NAMES[
        example["label"]
    ]
    for example in raw_dataset[
        validation_split
    ]
]


print(
    "Train examples:",
    len(tokenized_train)
)

print(
    "Validation examples:",
    len(tokenized_validation)
)

print(
    "Train batches:",
    len(train_dataloader)
)

print(
    "Validation batches:",
    len(eval_dataloader)
)

Train examples: 8551
Validation examples: 1043
Train batches: 268
Validation batches: 33


# Finding all linear layers

In [170]:
linear_modules = {}

for name, module in base_model.named_modules():

    if isinstance(module, nn.Linear):

        if "lm_head" not in name:

            linear_modules[name] = module

print("Number of target linear layers:", len(linear_modules))

for name, module in linear_modules.items():
    print(
        f"{name:70s}",
        tuple(module.weight.shape)
    )

Number of target linear layers: 192
encoder.block.0.layer.0.SelfAttention.q                                (768, 768)
encoder.block.0.layer.0.SelfAttention.k                                (768, 768)
encoder.block.0.layer.0.SelfAttention.v                                (768, 768)
encoder.block.0.layer.0.SelfAttention.o                                (768, 768)
encoder.block.0.layer.1.DenseReluDense.wi                              (3072, 768)
encoder.block.0.layer.1.DenseReluDense.wo                              (768, 3072)
encoder.block.1.layer.0.SelfAttention.q                                (768, 768)
encoder.block.1.layer.0.SelfAttention.k                                (768, 768)
encoder.block.1.layer.0.SelfAttention.v                                (768, 768)
encoder.block.1.layer.0.SelfAttention.o                                (768, 768)
encoder.block.1.layer.1.DenseReluDense.wi                              (3072, 768)
encoder.block.1.layer.1.DenseReluDense.wo                  

# Gradient probing

In [171]:
def run_gradient_probe(
    model,
    probe_dataloader,
    target_modules,
    n_batches=64,
    device="cuda",
):

    model = model.to(device)

    # We need training-like forward behavior during probing.
    model.train()

    # Freeze everything first.
    for param in model.parameters():
        param.requires_grad_(False)

    # Enable gradients ONLY for target linear weights.
    probe_params = {}

    for name, module in target_modules.items():

        module.weight.requires_grad_(True)

        probe_params[name] = module.weight

    # CPU gradient buffers.
    accumulated_grads = {
        name: torch.zeros_like(
            param,
            device="cpu",
            dtype=torch.float32,
        )
        for name, param in probe_params.items()
    }

    batches_used = 0

    for batch_idx, batch in enumerate(probe_dataloader):

        if batch_idx >= n_batches:
            break

        batches_used += 1

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Clear any old gradients.
        for param in probe_params.values():
            param.grad = None

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        # Copy gradients to CPU.
        for name, param in probe_params.items():

            if param.grad is not None:

                accumulated_grads[name] += (
                    param.grad.detach()
                    .float()
                    .cpu()
                )

                # Free GPU gradient.
                param.grad = None

        if (batch_idx + 1) % 8 == 0:
            print(
                f"Gradient probe: "
                f"{batch_idx + 1}/{n_batches}"
            )

    # Average gradients.
    for name in accumulated_grads:

        accumulated_grads[name] /= batches_used

    # Freeze everything again.
    for param in model.parameters():
        param.requires_grad_(False)

    print(
        f"\nGradient probing finished. "
        f"Batches used: {batches_used}"
    )

    return accumulated_grads

In [172]:
accumulated_grads = run_gradient_probe(
    model=base_model,
    probe_dataloader=train_dataloader,
    target_modules=linear_modules,
    n_batches=N_PROBE_BATCHES,
    device=device,
)

Gradient probe: 8/64
Gradient probe: 16/64
Gradient probe: 24/64
Gradient probe: 32/64
Gradient probe: 40/64
Gradient probe: 48/64
Gradient probe: 56/64
Gradient probe: 64/64

Gradient probing finished. Batches used: 64


In [173]:
print(
    "Number of accumulated gradients:",
    len(accumulated_grads)
)

first_name = next(iter(accumulated_grads))

print(
    first_name,
    accumulated_grads[first_name].shape,
    accumulated_grads[first_name].norm().item(),
)

Number of accumulated gradients: 192
encoder.block.0.layer.0.SelfAttention.q torch.Size([768, 768]) 0.05156303197145462


# GoRA importance calculation

In [174]:
def compute_gora_importance(
    model,
    accumulated_grads,
    target_modules,
):

    importance = {}

    for name, grad in accumulated_grads.items():

        weight = target_modules[name].weight.detach()

        weight_cpu = weight.float().cpu()

        importance[name] = (
            (weight_cpu * grad)
            .abs()
            .mean()
            .item()
        )

    return importance

In [175]:
importance = compute_gora_importance(
    base_model,
    accumulated_grads,
    linear_modules,
)

In [176]:
sorted_importance = sorted(
    importance.items(),
    key=lambda x: x[1],
    reverse=True,
)

print("\nTop 20 most important layers:\n")

for name, score in sorted_importance[:20]:

    print(
        f"{score:.8e}  {name}"
    )


Top 20 most important layers:

4.09447557e-06  decoder.block.10.layer.1.EncDecAttention.o
4.08922551e-06  encoder.block.10.layer.0.SelfAttention.v
3.66085555e-06  encoder.block.7.layer.0.SelfAttention.v
3.65249957e-06  encoder.block.8.layer.0.SelfAttention.v
3.48378990e-06  encoder.block.9.layer.0.SelfAttention.v
3.47759442e-06  encoder.block.5.layer.0.SelfAttention.v
3.43345118e-06  encoder.block.10.layer.0.SelfAttention.o
3.38324639e-06  encoder.block.6.layer.0.SelfAttention.v
3.36688436e-06  decoder.block.10.layer.1.EncDecAttention.v
3.23258337e-06  encoder.block.9.layer.0.SelfAttention.o
3.19384071e-06  encoder.block.4.layer.0.SelfAttention.v
3.17646686e-06  encoder.block.4.layer.0.SelfAttention.o
3.14684917e-06  decoder.block.9.layer.1.EncDecAttention.o
3.10398377e-06  encoder.block.11.layer.0.SelfAttention.v
3.06098900e-06  decoder.block.11.layer.0.SelfAttention.v
2.89534478e-06  encoder.block.0.layer.0.SelfAttention.o
2.84508269e-06  decoder.block.8.layer.1.EncDecAttention.o
2.

# GoRA rank allocation

In [177]:
def allocate_gora_ranks(
    target_modules,
    importance,
    reference_rank=8,
    min_rank=4,
    max_rank=32,
):

    # 1. Normalize importance

    total_importance = sum(
        importance.values()
    )

    advantages = {
        name: score / total_importance
        for name, score in importance.items()
    }

    # 2. Compute total parameter budget
    # b_i = (sqrt(m) + sqrt(n)) * r_ref

    total_budget = 0.0

    for name, module in target_modules.items():

        m, n = module.weight.shape

        total_budget += (
            math.sqrt(m)
            + math.sqrt(n)
        ) * reference_rank

    # 3. Allocate rank

    rank_pattern = {}

    for name, module in target_modules.items():

        m, n = module.weight.shape

        raw_rank = (
            total_budget
            * advantages[name]
            / (
                math.sqrt(m)
                + math.sqrt(n)
            )
        )

        rank = round(raw_rank)

        rank = max(
            min_rank,
            min(max_rank, rank)
        )

        rank_pattern[name] = rank

    return (
        rank_pattern,
        advantages,
        total_budget,
    )

In [178]:
rank_pattern, advantages, total_budget = allocate_gora_ranks(
    target_modules=linear_modules,
    importance=importance,
    reference_rank=REFERENCE_RANK,
    min_rank=MIN_RANK,
    max_rank=MAX_RANK,
)

In [179]:
rank_values = list(rank_pattern.values())

print("Number of adapted layers:", len(rank_values))

print(
    "Min rank:",
    min(rank_values)
)

print(
    "Max rank:",
    max(rank_values)
)

print(
    "Mean rank:",
    np.mean(rank_values)
)

print(
    "Reference rank:",
    REFERENCE_RANK
)

Number of adapted layers: 192
Min rank: 4
Max rank: 32
Mean rank: 9.713541666666666
Reference rank: 8


In [180]:
print("\nRank distribution:\n")

for name, rank in rank_pattern.items():

    print(
        f"r={rank:2d}  {name}"
    )


Rank distribution:

r=10  encoder.block.0.layer.0.SelfAttention.q
r=10  encoder.block.0.layer.0.SelfAttention.k
r=21  encoder.block.0.layer.0.SelfAttention.v
r=23  encoder.block.0.layer.0.SelfAttention.o
r= 7  encoder.block.0.layer.1.DenseReluDense.wi
r= 6  encoder.block.0.layer.1.DenseReluDense.wo
r=10  encoder.block.1.layer.0.SelfAttention.q
r= 9  encoder.block.1.layer.0.SelfAttention.k
r=21  encoder.block.1.layer.0.SelfAttention.v
r=20  encoder.block.1.layer.0.SelfAttention.o
r= 8  encoder.block.1.layer.1.DenseReluDense.wi
r= 5  encoder.block.1.layer.1.DenseReluDense.wo
r=11  encoder.block.2.layer.0.SelfAttention.q
r=11  encoder.block.2.layer.0.SelfAttention.k
r=21  encoder.block.2.layer.0.SelfAttention.v
r=20  encoder.block.2.layer.0.SelfAttention.o
r= 8  encoder.block.2.layer.1.DenseReluDense.wi
r= 5  encoder.block.2.layer.1.DenseReluDense.wo
r=12  encoder.block.3.layer.0.SelfAttention.q
r=11  encoder.block.3.layer.0.SelfAttention.k
r=21  encoder.block.3.layer.0.SelfAttention.v
r

# Count the resulting parameter budget

In [181]:
def count_lora_parameters(
    target_modules,
    rank_pattern,
):

    total = 0

    for name, module in target_modules.items():

        m, n = module.weight.shape

        r = rank_pattern[name]

        total += r * (m + n)

    return total

In [182]:
gora_parameter_count = count_lora_parameters(
    linear_modules,
    rank_pattern,
)

print(
    "GoRA trainable parameters:",
    f"{gora_parameter_count:,}"
)

GoRA trainable parameters: 3,408,384


# Build the GoRA PEFT model

In [183]:
def build_gora_model(
    base_model,
    rank_pattern,
):

    config = LoraConfig(
        r=REFERENCE_RANK,

        lora_alpha=LORA_ALPHA,

        lora_dropout=0.0,

        bias="none",

        task_type="SEQ_2_SEQ_LM",

        # All linear layers.
        target_modules="all-linear",

        # GoRA adaptive ranks.
        rank_pattern=rank_pattern,

        # GoRA uses sqrt(alpha/r) scaling.
        use_rslora=True,
    )

    model = get_peft_model(
        base_model,
        config,
    )

    return model

In [184]:
gora_model = build_gora_model(
    base_model,
    rank_pattern,
)

gora_model.print_trainable_parameters()

trainable params: 3,408,384 || all params: 226,311,936 || trainable%: 1.5061


In [185]:
@torch.no_grad()
def initialize_gora(
    peft_model,
    accumulated_grads,
    target_modules,
    gamma=0.05,
    lora_alpha=32,
):

    initialized = 0

    # Map original module names.
    original_module_dict = target_modules

    for name, module in peft_model.named_modules():

        if not hasattr(module, "lora_A"):
            continue

        # Find the corresponding original layer.
        matches = [
            original_name
            for original_name in accumulated_grads
            if name.endswith(original_name)
        ]

        if len(matches) == 0:
            continue

        original_name = matches[0]

        G = accumulated_grads[
            original_name
        ]

        # PEFT matrices
        #
        # A = [r, input]
        # B = [output, r]

        A = module.lora_A["default"].weight

        # Make sure gradient has same dtype/device.
        G = G.to(
            device=A.device,
            dtype=A.dtype,
        )

        # Kaiming initialization for A
        #
        # Paper:
        # A_paper ~ Kaiming Uniform
        #
        # PEFT A has shape [r, input].
        # We initialize it explicitly.

        nn.init.kaiming_uniform_(
            A,
            a=math.sqrt(5),
        )

        # Compute:
        #
        # B = -G A^T (A A^T)^(-1)
        #
        # This is equivalent to the paper's
        # pseudo-inverse formulation.

        AA_T = A @ A.T

        # More numerically stable than explicit inverse.
        pseudo_inverse = torch.linalg.pinv(
            AA_T
        )

        B = (
            -G
            @ A.T
            @ pseudo_inverse
        )

        # GoRA scaling:
        #
        # B <- gamma * sqrt(m) / alpha * B

        m = G.shape[0]

        scaling = (
            gamma
            * math.sqrt(m)
            / lora_alpha
        )

        B = B * scaling

        # Copy into PEFT LoRA B.
        module.lora_B[
            "default"
        ].weight.copy_(B)

        initialized += 1

    print(
        f"Initialized {initialized} GoRA layers."
    )

In [186]:
initialize_gora(
    peft_model=gora_model,
    accumulated_grads=accumulated_grads,
    target_modules=linear_modules,
    gamma=GAMMA,
    lora_alpha=LORA_ALPHA,
)

Initialized 192 GoRA layers.


In [187]:
checked = 0

for name, module in gora_model.named_modules():

    if hasattr(module, "lora_A"):

        A_norm = (
            module.lora_A["default"]
            .weight
            .data
            .norm()
            .item()
        )

        B_norm = (
            module.lora_B["default"]
            .weight
            .data
            .norm()
            .item()
        )

        print(
            name,
            "| A norm:",
            A_norm,
            "| B norm:",
            B_norm,
        )

        checked += 1

        if checked >= 10:
            break

base_model.model.encoder.block.0.layer.0.SelfAttention.q | A norm: 1.8159618377685547 | B norm: 0.00043317326344549656
base_model.model.encoder.block.0.layer.0.SelfAttention.k | A norm: 1.839874505996704 | B norm: 5.403810064308345e-05
base_model.model.encoder.block.0.layer.0.SelfAttention.v | A norm: 2.651141405105591 | B norm: 0.00018760778766591102
base_model.model.encoder.block.0.layer.0.SelfAttention.o | A norm: 2.7514424324035645 | B norm: 0.0001502270606579259
base_model.model.encoder.block.0.layer.1.DenseReluDense.wi | A norm: 1.5312907695770264 | B norm: 0.00016950361896306276
base_model.model.encoder.block.0.layer.1.DenseReluDense.wo | A norm: 1.418175458908081 | B norm: 6.279967055888847e-05
base_model.model.encoder.block.1.layer.0.SelfAttention.q | A norm: 1.8337767124176025 | B norm: 0.0004205317236483097
base_model.model.encoder.block.1.layer.0.SelfAttention.k | A norm: 1.7386715412139893 | B norm: 5.031388354836963e-05
base_model.model.encoder.block.1.layer.0.SelfAttenti

In [188]:
print("\nActual PEFT ranks:\n")

for name, module in gora_model.named_modules():

    if hasattr(module, "lora_A"):

        actual_rank = (
            module.lora_A["default"]
            .weight
            .shape[0]
        )

        print(
            f"r={actual_rank:2d}  {name}"
        )


Actual PEFT ranks:

r=10  base_model.model.encoder.block.0.layer.0.SelfAttention.q
r=10  base_model.model.encoder.block.0.layer.0.SelfAttention.k
r=21  base_model.model.encoder.block.0.layer.0.SelfAttention.v
r=23  base_model.model.encoder.block.0.layer.0.SelfAttention.o
r= 7  base_model.model.encoder.block.0.layer.1.DenseReluDense.wi
r= 6  base_model.model.encoder.block.0.layer.1.DenseReluDense.wo
r=10  base_model.model.encoder.block.1.layer.0.SelfAttention.q
r= 9  base_model.model.encoder.block.1.layer.0.SelfAttention.k
r=21  base_model.model.encoder.block.1.layer.0.SelfAttention.v
r=20  base_model.model.encoder.block.1.layer.0.SelfAttention.o
r= 8  base_model.model.encoder.block.1.layer.1.DenseReluDense.wi
r= 5  base_model.model.encoder.block.1.layer.1.DenseReluDense.wo
r=11  base_model.model.encoder.block.2.layer.0.SelfAttention.q
r=11  base_model.model.encoder.block.2.layer.0.SelfAttention.k
r=21  base_model.model.encoder.block.2.layer.0.SelfAttention.v
r=20  base_model.model.enc

In [189]:
for name, module in gora_model.named_modules():

    if hasattr(module, "scaling"):

        print(
            name,
            module.scaling
        )

        break

base_model.model.encoder.block.0.layer.0.SelfAttention 1.0


# Evaluation function

In [190]:
@torch.no_grad()
def evaluate(
    model,
    eval_dataloader,
    tokenizer,
    true_labels_text,
    device,
):

    model.eval()

    all_predictions = []

    for batch in eval_dataloader:

        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=8,
        )

        decoded = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True,
        )

        cleaned = [
            prediction.strip()
            for prediction in decoded
        ]

        all_predictions.extend(
            cleaned
        )

    metric = compute_metrics(
        all_predictions,
        true_labels_text,
    )

    return metric, all_predictions

In [191]:
def compute_metrics(
    predictions,
    true_labels_text,
):

    label_to_id = {
        v: k
        for k, v in LABEL_NAMES.items()
    }

    y_true = [
        label_to_id[x]
        for x in true_labels_text
    ]

    y_pred = [
        label_to_id.get(x, -1)
        for x in predictions
    ]

    if config["metric"] == "matthews":

        metric_value = matthews_corrcoef(
            y_true,
            y_pred,
        )

    else:

        metric_value = accuracy_score(
            y_true,
            y_pred,
        )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted", # Use 'weighted' for imbalanced classes
    )

    return metric_value, f1

# GoRA optimizer

In [192]:
from sklearn.metrics import (
    accuracy_score,
    matthews_corrcoef,
    f1_score,
)

In [193]:
def create_gora_optimizer(
    model,
    peak_lr=1e-4,
):

    A_params = []
    B_params = []
    other_params = []

    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        if "lora_A" in name:

            A_params.append(param)

        elif "lora_B" in name:

            B_params.append(param)

        else:

            other_params.append(param)

    print(
        "A parameters:",
        sum(
            p.numel()
            for p in A_params
        ),
    )

    print(
        "B parameters:",
        sum(
            p.numel()
            for p in B_params
        ),
    )

    print(
        "Other trainable parameters:",
        sum(
            p.numel()
            for p in other_params
        ),
    )

    parameter_groups = [
        {
            "params": A_params,
            "lr": peak_lr,
        },

        {
            "params": B_params,
            "lr": peak_lr * 16,
        },
    ]

    if len(other_params) > 0:

        parameter_groups.append(
            {
                "params": other_params,
                "lr": peak_lr,
            }
        )

    optimizer = Adam(
        parameter_groups,
        betas=(BETA1, BETA2),
        weight_decay=WEIGHT_DECAY,
    )

    return optimizer

# GoRA training

In [194]:
def train_gora(
    model,
    train_dataloader,
    eval_dataloader,
    tokenizer,
    true_labels_text,
    device,
):

    model = model.to(device)

    total_steps = (
        len(train_dataloader)
        * NUM_EPOCHS
    )

    warmup_steps = math.ceil(
        total_steps
        * WARMUP_RATIO
    )

    print(
        "Total training steps:",
        total_steps,
    )

    print(
        "Warmup steps:",
        warmup_steps,
    )

    optimizer = create_gora_optimizer(
        model,
        peak_lr=PEAK_LR,
    )

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    if torch.cuda.is_available():

        torch.cuda.reset_peak_memory_stats(
            device
        )

    train_start = time.time()

    for epoch in range(NUM_EPOCHS):

        model.train()

        epoch_loss = 0.0

        for step, batch in enumerate(
            train_dataloader
        ):

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            optimizer.zero_grad(
                set_to_none=True
            )

            outputs = model(
                **batch
            )

            loss = outputs.loss

            loss.backward()

            optimizer.step()

            scheduler.step()

            epoch_loss += loss.item()

        avg_loss = (
            epoch_loss
            / len(train_dataloader)
        )

        print(
            f"Epoch {epoch + 1}/{NUM_EPOCHS} "
            f"| loss = {avg_loss:.4f}"
        )

    train_time = (
        time.time()
        - train_start
    )

    accuracy, predictions = evaluate(
        model,
        eval_dataloader,
        tokenizer,
        true_labels_text,
        device,
    )

    accuracy, f1 = compute_metrics(
        predictions,
        true_labels_text,
    )

    if torch.cuda.is_available():

        peak_memory = (
            torch.cuda.max_memory_allocated(
                device
            )
            / 1e9
        )

    else:

        peak_memory = 0.0

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    print("\n==============================")
    print("GoRA Results")
    print("==============================")

    print(
        f"Accuracy: {accuracy:.4f}"
    )

    print(
        f"F1:       {f1:.4f}"
    )

    print(
        f"Trainable parameters: "
        f"{trainable_params:,}"
    )

    print(
        f"Trainable %: "
        f"{100 * trainable_params / total_params:.4f}%"
    )

    print(
        f"Training time: "
        f"{train_time:.1f}s"
    )

    print(
        f"Peak GPU memory: "
        f"{peak_memory:.2f} GB"
    )

    return {
        "accuracy": accuracy,
        "f1": f1,
        "train_time": train_time,
        "peak_memory_gb": peak_memory,
        "trainable_params": trainable_params,
    }

In [195]:
# RUN CURRENT DATASET

print("\n")
print("=" * 70)
print(
    f"EXPERIMENT: {DATASET_NAME.upper()}"
)
print("=" * 70)


# GoRA

gora_model = build_gora_model(
    base_model,
    rank_pattern,
)

gora_model.print_trainable_parameters()






EXPERIMENT: COLA


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 3,408,384 || all params: 226,311,936 || trainable%: 1.5061


In [196]:
gora_results = train_gora(
    model=gora_model,
    train_dataloader=train_dataloader,
    eval_dataloader=eval_dataloader,
    tokenizer=tokenizer,
    true_labels_text=true_labels_text,
    device=device,
)

Total training steps: 268
Warmup steps: 9
A parameters: 1665024
B parameters: 1743360
Other trainable parameters: 0
Epoch 1/1 | loss = 0.2272

GoRA Results
Accuracy: 0.5774
F1:       0.8204
Trainable parameters: 3,408,384
Trainable %: 1.5061%
Training time: 196.1s
Peak GPU memory: 7.24 GB


# LoRA

In [197]:
lora_base = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

lora_config = LoraConfig(
    r=REFERENCE_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    target_modules="all-linear",
)

lora_model = get_peft_model(
    lora_base,
    lora_config,
)

lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

trainable params: 3,244,032 || all params: 226,147,584 || trainable%: 1.4345


In [198]:
def train_plain_lora(
    model,
    train_dataloader,
    eval_dataloader,
    tokenizer,
    true_labels_text,
    device,
):

    model = model.to(device)

    total_steps = (
        len(train_dataloader)
        * NUM_EPOCHS
    )

    warmup_steps = math.ceil(
        total_steps
        * WARMUP_RATIO
    )

    optimizer = Adam(
        model.parameters(),
        lr=PEAK_LR,
        betas=(BETA1, BETA2),
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    if torch.cuda.is_available():

        torch.cuda.reset_peak_memory_stats(
            device
        )

    start = time.time()

    for epoch in range(NUM_EPOCHS):

        model.train()

        epoch_loss = 0.0

        for batch in train_dataloader:

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            optimizer.zero_grad(
                set_to_none=True
            )

            loss = model(
                **batch
            ).loss

            loss.backward()

            optimizer.step()

            scheduler.step()

            epoch_loss += loss.item()

        print(
            f"LoRA epoch {epoch + 1} "
            f"| loss = "
            f"{epoch_loss / len(train_dataloader):.4f}"
        )

    train_time = (
        time.time() - start
    )

    accuracy, predictions = evaluate(
        model,
        eval_dataloader,
        tokenizer,
        true_labels_text,
        device,
    )

    accuracy, f1 = compute_metrics(
        predictions,
        true_labels_text,
    )

    print("\n==============================")
    print("Plain LoRA Results")
    print("==============================")

    print(
        f"Accuracy: {accuracy:.4f}"
    )

    print(
        f"F1: {f1:.4f}"
    )

    return {
        "accuracy": accuracy,
        "f1": f1,
        "train_time": train_time,
    }

In [199]:
lora_results = train_plain_lora(
    model=lora_model,
    train_dataloader=train_dataloader,
    eval_dataloader=eval_dataloader,
    tokenizer=tokenizer,
    true_labels_text=true_labels_text,
    device=device,
)

LoRA epoch 1 | loss = 0.2142

Plain LoRA Results
Accuracy: 0.5259
F1: 0.7942


# Zero-shot baseline

In [213]:
from collections import Counter

print(
    "\nvalidation distribution:"
)

print(
    Counter(true_labels_text)
)


validation distribution:
Counter({'acceptable': 721, 'unacceptable': 322})


In [201]:
zero_shot_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

zero_shot_model.eval()

start = time.time()

zs_accuracy, zs_predictions = evaluate(
    zero_shot_model,
    eval_dataloader,
    tokenizer,
    true_labels_text,
    device,
)

zs_time = time.time() - start

zs_accuracy, zs_f1 = compute_metrics(
    zs_predictions,
    true_labels_text,
)

print(
    f"Zero-shot accuracy: "
    f"{zs_accuracy:.4f}"
)

print(
    f"Zero-shot F1: "
    f"{zs_f1:.4f}"
)

print(
    "Sample predictions:",
    zs_predictions[:10],
)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Zero-shot accuracy: 0.4867
Zero-shot F1: 0.7716
Sample predictions: ['acceptable', 'acceptable', 'acceptable', 'acceptable', 'acceptable', 'acceptable', 'acceptable', 'acceptable', 'acceptable', 'acceptable']


In [202]:
print(
    "\nZero-shot prediction distribution:"
)

print(
    Counter(zs_predictions)
)


Zero-shot prediction distribution:
Counter({'acceptable': 879, 'unacceptable': 164})


In [203]:
for i in range(20):

    print(
        f"{i:02d} | "
        f"TRUE: {true_labels_text[i]} | "
        f"PRED: {zs_predictions[i]}"
    )

00 | TRUE: acceptable | PRED: acceptable
01 | TRUE: acceptable | PRED: acceptable
02 | TRUE: acceptable | PRED: acceptable
03 | TRUE: acceptable | PRED: acceptable
04 | TRUE: unacceptable | PRED: acceptable
05 | TRUE: unacceptable | PRED: acceptable
06 | TRUE: unacceptable | PRED: acceptable
07 | TRUE: acceptable | PRED: acceptable
08 | TRUE: acceptable | PRED: acceptable
09 | TRUE: acceptable | PRED: acceptable
10 | TRUE: acceptable | PRED: acceptable
11 | TRUE: acceptable | PRED: acceptable
12 | TRUE: acceptable | PRED: acceptable
13 | TRUE: unacceptable | PRED: acceptable
14 | TRUE: acceptable | PRED: acceptable
15 | TRUE: acceptable | PRED: acceptable
16 | TRUE: unacceptable | PRED: acceptable
17 | TRUE: acceptable | PRED: acceptable
18 | TRUE: acceptable | PRED: acceptable
19 | TRUE: acceptable | PRED: acceptable


# Final comparison

In [204]:
print("\n")
print("=" * 70)
print(
    f"{DATASET_NAME.upper()} RESULTS"
)
print("=" * 70)

print(
    f"Metric: {config['metric']}"
)

print("-" * 70)

print(
    f"Zero-shot: "
    f"{zs_accuracy:.4f}"
)

print(
    f"Plain LoRA: "
    f"{lora_results['accuracy']:.4f}"
)

print(
    f"GoRA: "
    f"{gora_results['accuracy']:.4f}"
)

print("=" * 70)



COLA RESULTS
Metric: matthews
----------------------------------------------------------------------
Zero-shot: 0.4867
Plain LoRA: 0.5259
GoRA: 0.5774


# EXPERIMENT RESULTS

In [ ]:
# print("\n")
# print("=" * 70)
# print(
#     f"{DATASET_NAME.upper()} RESULTS"
# )
# print("=" * 70)

# print(
#     f"Metric: {config['metric']}"
# )

# print("-" * 70)

# print(
#     f"Zero-shot: "
#     f"{zs_accuracy:.4f}"
# )

# print(
#     f"Plain LoRA: "
#     f"{lora_results['accuracy']:.4f}"
# )

# print(
#     f"GoRA: "
#     f"{gora_results['accuracy']:.4f}"
# )

# print("=" * 70)



SST2 RESULTS
Metric: accuracy
----------------------------------------------------------------------
Zero-shot: 0.9392
Plain LoRA: 0.9404
GoRA: 0.9335


In [ ]:
# print("\n")
# print("=" * 70)
# print(
#     f"{DATASET_NAME.upper()} RESULTS"
# )
# print("=" * 70)

# print(
#     f"Metric: {config['metric']}"
# )

# print("-" * 70)

# print(
#     f"Zero-shot: "
#     f"{zs_accuracy:.4f}"
# )

# print(
#     f"Plain LoRA: "
#     f"{lora_results['accuracy']:.4f}"
# )

# print(
#     f"GoRA: "
#     f"{gora_results['accuracy']:.4f}"
# )

# print("=" * 70)



COLA RESULTS
Metric: matthews
----------------------------------------------------------------------
Zero-shot: 0.4867
Plain LoRA: 0.5259
GoRA: 0.5774


In [ ]:
# print("\n")
# print("=" * 70)
# print(
#     f"{DATASET_NAME.upper()} RESULTS"
# )
# print("=" * 70)

# print(
#     f"Metric: {config['metric']}"
# )

# print("-" * 70)

# print(
#     f"Zero-shot: "
#     f"{zs_accuracy:.4f}"
# )

# print(
#     f"Plain LoRA: "
#     f"{lora_results['accuracy']:.4f}"
# )

# print(
#     f"GoRA: "
#     f"{gora_results['accuracy']:.4f}"
# )

# print("=" * 70)



MRPC RESULTS
Metric: accuracy
----------------------------------------------------------------------
Zero-shot: 0.8578
Plain LoRA: 0.8725
GoRA: 0.8873


In [ ]:
# print("\n")
# print("=" * 70)
# print(
#     f"{DATASET_NAME.upper()} RESULTS"
# )
# print("=" * 70)

# print(
#     f"Metric: {config['metric']}"
# )

# print("-" * 70)

# print(
#     f"Zero-shot: "
#     f"{zs_accuracy:.4f}"
# )

# print(
#     f"Plain LoRA: "
#     f"{lora_results['accuracy']:.4f}"
# )

# print(
#     f"GoRA: "
#     f"{gora_results['accuracy']:.4f}"
# )

# print("=" * 70)



RTE RESULTS
Metric: accuracy
----------------------------------------------------------------------
Zero-shot: 0.3357
Plain LoRA: 0.4152
GoRA: 0.5812


In [206]:
all_results[DATASET_NAME] = {
    "Zero-shot": zs_accuracy,
    "LoRA": lora_results["accuracy"],
    "GoRA": gora_results["accuracy"],
    "LoRA trainable params": sum(
        p.numel()
        for p in lora_model.parameters()
        if p.requires_grad
    ),
    "GoRA trainable params": gora_results["trainable_params"],
    "GoRA training time (s)": gora_results["train_time"],
    "GoRA peak memory (GB)": gora_results["peak_memory_gb"],
}

In [207]:
print(all_results.keys())

dict_keys(['mrpc', 'rte', 'sst2', 'cola'])


In [208]:
import pandas as pd

results_df = pd.DataFrame.from_dict(
    all_results,
    orient="index"
)

results_df.index.name = "Dataset"

# Convert scores from 0-1 to percentages
results_df[
    ["Zero-shot", "LoRA", "GoRA"]
] = (
    results_df[
        ["Zero-shot", "LoRA", "GoRA"]
    ] * 100
)

display(
    results_df[
        ["Zero-shot", "LoRA", "GoRA"]
    ].round(2)
)

,Zero-shot,LoRA,GoRA
Dataset,,,
mrpc,85.78,87.25,88.73
rte,33.57,41.52,58.12
sst2,93.92,94.04,93.35
cola,48.67,52.59,57.74


In [209]:
paper_results = {
    "mnli": {
        "LoRA": 85.30,
        "GoRA": 85.91,
    },

    "sst2": {
        "LoRA": 94.04,
        "GoRA": 94.68,
    },

    "cola": {
        "LoRA": 69.35,
        "GoRA": 79.86,
    },

    "qnli": {
        "LoRA": 92.96,
        "GoRA": 93.27,
    },

    "mrpc": {
        "LoRA": 68.38,
        "GoRA": 86.10,
    },
}

In [210]:
comparison_rows = []

for dataset, result in all_results.items():

    row = {
        "Dataset": dataset.upper(),

        "Zero-shot (ours)":
            result["Zero-shot"] * 100,

        "LoRA (ours)":
            result["LoRA"] * 100,

        "GoRA (ours)":
            result["GoRA"] * 100,

        "LoRA (paper)":
            paper_results.get(dataset, {}).get(
                "LoRA", None
            ),

        "GoRA (paper)":
            paper_results.get(dataset, {}).get(
                "GoRA", None
            ),
    }

    comparison_rows.append(row)


comparison_df = pd.DataFrame(
    comparison_rows
)

display(
    comparison_df.round(2)
)

,Dataset,Zero-shot (ours),LoRA (ours),GoRA (ours),LoRA (paper),GoRA (paper)
0,MRPC,85.78,87.25,88.73,68.38,86.10
1,RTE,33.57,41.52,58.12,NaN,NaN
2,SST2,93.92,94.04,93.35,94.04,94.68
3,COLA,48.67,52.59,57.74,69.35,79.86


In [211]:
comparison_df["GoRA gain (ours)"] = (
    comparison_df["GoRA (ours)"]
    - comparison_df["LoRA (ours)"]
)

comparison_df["GoRA gain (paper)"] = (
    comparison_df["GoRA (paper)"]
    - comparison_df["LoRA (paper)"]
)

display(
    comparison_df.round(2)
)

,Dataset,Zero-shot (ours),LoRA (ours),GoRA (ours),LoRA (paper),GoRA (paper),GoRA gain (ours),GoRA gain (paper)
0,MRPC,85.78,87.25,88.73,68.38,86.10,1.47,17.72
1,RTE,33.57,41.52,58.12,NaN,NaN,16.61,NaN
2,SST2,93.92,94.04,93.35,94.04,94.68,-0.69,0.64
3,COLA,48.67,52.59,57.74,69.35,79.86,5.15,10.51


In [212]:
efficiency_df = pd.DataFrame.from_dict(
    all_results,
    orient="index"
)

efficiency_df.index.name = "Dataset"

display(
    efficiency_df[
        [
            "LoRA trainable params",
            "GoRA trainable params",
            "GoRA training time (s)",
            "GoRA peak memory (GB)",
        ]
    ].round(2)
)

,LoRA trainable params,GoRA trainable params,GoRA training time (s),GoRA peak memory (GB)
Dataset,,,,
mrpc,3244032,3266304,82.90,5.29
rte,3244032,3273216,60.05,7.22
sst2,3244032,3519744,1548.08,7.26
cola,3244032,3408384,196.10,7.24
